[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/scenegraph/scenegraph.ipynb)

# A Scenegraph, a Robot Arm and a Camera, as One Map

This example poses a five-part robot arm and photographs it through a two-lens camera. In a graphics pipeline every vertex goes through the same chain of steps: stretch a unit box into the shape of a part, place it where the arm's joints put it, move it into the camera's frame, send a ray from it through the lenses onto the sensor, and scale the sensor to pixels. The steps compose into one transformation per part before any vertex is touched, so each vertex costs a single application.

Every step here is a map in projective geometric algebra. A part's stretch is a map on points, `Point <- Point`; a motor places it, `pose >> scale`; the camera joins each point with the pupil into a ray, a map from points to lines, `Point & pupil`; a thin lens is its formula on rays, `Line - (center & (Line ^ plane)) / focal_length`, a map from lines to lines; the sensor meets each ray in a point, `Line ^ sensor`. Calling one map on another composes them, so the whole pipeline collapses to one map from a part's unit box to pixels.

In the matrix notation of computer graphics the collapsed map reads as the product of the viewport, projection, view and model matrices, $V P C M_i$, one per part, and in ray-transfer optics a thin lens as the matrix $\begin{pmatrix} 1 & 0 \\ -1/f & 1 \end{pmatrix}$ on a ray's height and angle.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.geometry.scenegraph import core, render
from examples.geometry.scenegraph.core import canonical_unit_box

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                      # Vector+
Line = ga.gatype.antibivector()                 # Antibivector-
Point = ga.gatype.antivector()                  # Antivector-
Motor = ga.gatype.rotor()                       # Rotor+
PointMap = ga.gatype((Point, Point))
LineMap = ga.gatype((Line, Line))
origin = mv.w.dual().normalized()               # [] Point-
optical_axis = (origin & mv.z.dual()).normalized()  # [] Line-

## 1. Unit boxes, stretched and placed

Every part of the arm is one unit cube, stretched by a map on points that sends each coordinate to its scaled axis and keeps the weight, then placed by the motor of its joint. Each joint's pivot is the product of the motors before it: a lift and a yaw at the base, then a lift and a pitch at the shoulder, the elbow and the wrist. A motor applied to a map moves what the map returns: the part still takes points of the unit cube, but returns them stretched and placed.

In [ ]:
# The arm's pose: base yaw, shoulder, elbow and wrist pitch.
pose_angles = (0.35, -0.45, 0.85, -0.40)

In [ ]:
def anisotropic_scale(scale_x: float, scale_y: float, scale_z: float) -> PointMap:
    """The anisotropic scaling map stretches points along the principal axes while preserving their weights."""
    # The coordinate planes read a point's coordinates, mv.x & point being its x; each
    # coordinate is sent to its scaled axis direction, and the weight to the origin.
    coordinate_planes = mv.vector(np.eye(4))                                   # [coordinates] Plane+
    scaled_axes = mv.antivector(np.diag([scale_x, scale_y, scale_z, 1.0]))     # [coordinates] Point-
    return (scaled_axes * (coordinate_planes & Point)).sum(axis=0)             # [] Point- <- Point-


def robot_arm(joint_angles: tuple) -> PointMap:
    """Each body of the five-part arm is the canonical unit box scaled to shape and placed into the world by its cumulative joint motor."""
    theta_base, theta_shoulder, theta_elbow, theta_wrist = joint_angles

    # Body 0: Base pedestal resting on the floor z == 0
    body_0 = (mv.zw * 0.125 / 2).exp() >> anisotropic_scale(0.9, 0.9, 0.25)    # [] Point- <- Point-

    # Turret base pivot at height 0.25, yawing around the vertical z-axis:
    pivot_turret = (mv.zw * 0.25 / 2).exp() * (mv.xy * theta_base / 2).exp()   # [] Motor+
    body_1 = (pivot_turret * (mv.zw * 0.175 / 2).exp()) >> anisotropic_scale(0.5, 0.5, 0.35)

    # Shoulder pivot 0.35 above turret base, pitching around the local y-axis:
    pivot_shoulder = pivot_turret * (mv.zw * 0.35 / 2).exp() * (mv.zx * theta_shoulder / 2).exp()
    body_2 = (pivot_shoulder * (mv.zw * 0.6 / 2).exp()) >> anisotropic_scale(0.24, 0.24, 1.2)

    # Elbow pivot at the tip of the upper arm (1.2 units along it), pitching around the local y-axis:
    pivot_elbow = pivot_shoulder * (mv.zw * 1.2 / 2).exp() * (mv.zx * theta_elbow / 2).exp()
    body_3 = (pivot_elbow * (mv.zw * 0.5 / 2).exp()) >> anisotropic_scale(0.18, 0.18, 1.0)

    # Wrist pivot at the tip of the forearm (1.0 unit along it), pitching around the local y-axis:
    pivot_wrist = pivot_elbow * (mv.zw * 1.0 / 2).exp() * (mv.zx * theta_wrist / 2).exp()
    body_4 = (pivot_wrist * (mv.zw * 0.15 / 2).exp()) >> anisotropic_scale(0.28, 0.14, 0.3)

    return stack([body_0, body_1, body_2, body_3, body_4])                     # [bodies] Point- <- Point-


unit_box = canonical_unit_box()                                                # [vertices] Point-
bodies_to_world = robot_arm(pose_angles)                                       # [bodies] Point- <- Point-
world_vertices = bodies_to_world[:, None](unit_box[None, :])                   # [bodies, vertices] Point-

## 2. A two-lens camera

A thin lens shears each ray by its incidence with the lens's centre; the relay lens is the same map carried a little down the axis by a motor. The camera pulls world points into its own frame, joins each with a point of the pupil into a ray, refracts the ray through both lenses and meets it with the sensor. A viewport, a stretch and a shift, maps the sensor's physical size to its pixels, rows counted downward.

In [ ]:
# The camera's eye and target, the lenses' focal lengths and the relay's place, the pupil point's offset,
# the sensor's distance, and the sensor's size and pixels.
eye, target = mv.point([0.0, -3.6, 1.3]), mv.point([0.0, 0.0, 1.18])
front_focal, rear_focal, rear_gap = 1.0, 0.8, -0.3
pupil_offset, sensor_distance = 0.02, 1.25
sensor_size, pixels = (1.6, 1.2), (640, 480)

In [ ]:
def look_at(eye: Point, target: Point) -> Motor:
    """The camera motor places the origin at `eye` and turns the optical axis onto `eye & target`."""
    eye = eye.normalized()                                                     # [] Point-
    translator = (eye / origin).square_root()                                  # [] Motor+
    sight = (eye & target).normalized()                                        # [] Line-
    return (sight / (translator >> optical_axis)).square_root() * translator   # [] Motor+


def thin_lens(center: Point, plane: Plane, focal_length: float) -> LineMap:
    """A thin lens shears each ray proportionally to its incidence with the optical centre."""
    return Line - (center & (Line ^ plane)) / focal_length                     # [] Line- <- Line-


camera_pose = look_at(eye, target)                                             # [] Motor+

# The front lens sits at the origin; the rear lens is placed along -z by its displacement motor.
front_plane = -mv.z                                                            # [] Plane+
front_lens = thin_lens(origin, front_plane, front_focal)                       # [] Line- <- Line-
rear_placement = (mv.zw * rear_gap / 2).exp()                                  # [] Motor+
rear_plane = rear_placement >> front_plane                                     # [] Plane+
rear_lens = rear_placement >> thin_lens(origin, front_plane, rear_focal)(rear_placement << Line)   # [] Line- <- Line-

# Transform world points into local camera space, join with the pupil into rays, refract through both lenses, and meet the sensor.
pupil = (mv.xw * pupil_offset).exp() >> origin                                 # [] Point-
sensor_plane = mv.z + sensor_distance * mv.w                                   # [] Plane+
incoming_rays = (camera_pose << Point) & pupil                                 # [] Line- <- Point-
camera = rear_lens(front_lens(incoming_rays)) ^ sensor_plane                   # [] Point- <- Point-

# Map the physical sensor onto pixel coordinates (u, v), with rows increasing downward.
sensor_scale = anisotropic_scale(pixels[0] / sensor_size[0], -pixels[1] / sensor_size[1], 1.0)   # [] Point- <- Point-
viewport = ((mv.xw * pixels[0] / 2 + mv.yw * pixels[1] / 2) / 2).exp() >> sensor_scale           # [] Point- <- Point-
world_to_pixel = viewport(camera)                                              # [] Point- <- Point-

## 3. The whole pipeline as one map

Composing the world-to-pixel map with each part's placement gives one map per part, from the unit cube straight to pixels, and every vertex of every part goes through it in one pass. In the figure, the left panel shows the arm in the world with the camera and three rays traced from gripper corners through both lenses to the sensor, and the right panel shows the photograph on the sensor.

In [ ]:
traced_corners = [7, 6, 2]

# Collapse world-to-pixel and each body's placement into one map per body, and apply it to the unit box.
local_to_pixel = world_to_pixel(bodies_to_world)                               # [bodies] Point- <- Point-
raw_pixels = local_to_pixel[:, None](unit_box[None, :])                        # [bodies, vertices] Point-
projected_pixels = raw_pixels / (mv.w & raw_pixels)                            # [bodies, vertices] Point-

# Trace rays from the chosen gripper corners through the pupil, both lenses, and the sensor back into the world.
def trace_rays(scene_points: Point) -> tuple[Point, Point, Point, Point]:
    """Each ray from a scene point through the pupil crosses the front lens, the rear lens plane, and the sensor plane at these world points."""
    ray_in = (camera_pose << scene_points) & pupil                             # [rays] Line-
    ray_mid = front_lens(ray_in)                                               # [rays] Line-
    rear_hit = ray_mid ^ rear_plane                                            # [rays] Point-
    sensor_hit = rear_lens(ray_mid) ^ sensor_plane                             # [rays] Point-
    return (
        scene_points,
        (camera_pose >> pupil).broadcast_to(scene_points.shape),
        camera_pose >> (rear_hit / (mv.w & rear_hit)),
        camera_pose >> (sensor_hit / (mv.w & sensor_hit)),
    )


rays = trace_rays(world_vertices[-1, traced_corners])
save_figure(render.draw_scenegraph(world_vertices, projected_pixels, camera_pose, rays), "scenegraph");

## 4. The arm in motion

Over a loop of joint angles the arm swings and the camera photographs each pose through the same collapsed pipeline, a new map per part per frame, with the ray from the gripper's tip traced through the lenses.

In [ ]:
# Each joint's middle and swing over the loop, the frames, and the duration of each in milliseconds.
sweep_middles = np.array([0.0, -0.40, 0.85, -0.45])
sweep_swings = np.array([0.45, 0.25, 0.35, 0.25])
n_frames = 36
frame_duration = 60

In [ ]:
def robot_sweep(n_frames: int):
    """Yield each frame of a looping joint trajectory photographed through the compound camera."""
    for phase in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        # The base and elbow swing with the sine, the shoulder with the cosine, the wrist twice as fast.
        modulations = np.array([np.sin(phase), np.cos(phase), np.sin(phase), np.cos(2 * phase)])
        placed = robot_arm(tuple(sweep_middles + sweep_swings * modulations))   # [bodies] Point- <- Point-
        raw = world_to_pixel(placed)[:, None](unit_box[None, :])               # [bodies, vertices] Point-
        vertices = placed[:, None](unit_box[None, :])                          # [bodies, vertices] Point-
        yield vertices, raw / (mv.w & raw), camera_pose, trace_rays(vertices[-1, traced_corners[:1]])


# README picture: this makes plots/scenegraph.gif, tuned as a visual for the README. Don't change it; to
# experiment, copy it.
show_animation(render.animate_scenegraph(robot_sweep(n_frames)), "scenegraph", frame_duration)

In [ ]:
# checks
# The pipeline laid out here matches the core's.
np.testing.assert_allclose((bodies_to_world - core.robot_arm(pose_angles)[0]).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((camera_pose - core.look_at(eye, target)).kernel, 0.0, atol=1e-12)
# The collapsed map agrees with applying kinematics, camera and viewport in turn.
sequential = world_to_pixel(world_vertices)
sequential = sequential / (mv.w & sequential)
assert (sequential - projected_pixels).dual().norm_squared().to_array().max() < 1e-16